In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────
# Data files
HOUSING_SOURCE_FILE = "./housing_source.jsonl"
FLAT_OUTPUT_FILE = "./finetune_qa.jsonl"
MESSAGES_OUTPUT_FILE = "./finetune_messages.jsonl"

# Model configuration
BASE_MODEL_ID = "meta-llama/Llama-3.2-1B-instruct"
OUTPUT_DIR = "./llama3-lora-finetuned"
MERGED_MODEL_DIR = "./llama3-housing-lora-merged"

# Hugging Face Hub (optional - for pushing the merged model)
HF_MODEL_REPO = "SigwiLi/llama3-housing-lora-finetuned"

# Training hyperparameters
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRAD_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 2048

# Llama 3.2 LoRA Fine-tuning for Housing Market Assistant

This notebook fine-tunes **Llama 3.2 1B-Instruct** using QLoRA on a custom housing market dataset, then merges the LoRA weights into the base model.

## What this does:
1. **Data Preparation**: Converts JSONL source data into instruction-following format
2. **QLoRA Training**: Efficiently fine-tunes using 4-bit quantization + LoRA
3. **Merging**: Combines LoRA weights with base model for inference
4. **Publishing**: Pushes the merged model to Hugging Face Hub

## Requirements:
- GPU with sufficient VRAM (~8GB+ recommended)
- Source data in `housing_source.jsonl`
- HF Hub credentials for pushing (optional)


## Step 1: Data Preparation

Convert raw source data into instruction-following pairs and ChatML message format.

In [ ]:
from datasets import load_dataset
import json
import re
from pathlib import Path

print(f"Loading source data from: {HOUSING_SOURCE_FILE}")

SYSTEM_PROMPT = "You are a helpful housing market assistant. Answer only with information supported by the source."

def clean_text(text):
    """Remove excessive whitespace from text."""
    return re.sub(r"\s+", " ", text).strip()

def add_pair(pairs, instruction, output, source_id):
    """Add a Q&A pair to the list."""
    pairs.append({
        "instruction": instruction.strip(),
        "input": "",
        "output": clean_text(output),
        "source_id": source_id
    })

def generate_pairs_from_record(record):
    """Generate instruction-following pairs from a source record."""
    text = record["text"]
    source_id = record["id"]
    pairs = []

    if source_id == "housing_zillow_001":
        add_pair(pairs, "What was the Zillow Home Value Index for Iowa City, IA as of February 28, 2026?",
                 "As of February 28, 2026, the Zillow Home Value Index for Iowa City, IA was $292,103.", source_id)
        add_pair(pairs, "What was the year-over-year change in the Iowa City Zillow Home Value Index as of February 28, 2026?",
                 "The Iowa City Zillow Home Value Index was up 4.6 percent year over year as of February 28, 2026.", source_id)
        add_pair(pairs, "What was the median sale price in Iowa City as of January 31, 2026?",
                 "The median sale price in Iowa City was $289,417 as of January 31, 2026.", source_id)
        add_pair(pairs, "What was the median list price in Iowa City in the Zillow citywide overview?",
                 "The median list price in Iowa City was $297,567.", source_id)
        add_pair(pairs, "What was the median sale-to-list ratio in Iowa City?",
                 "The median sale-to-list ratio in Iowa City was 0.983, or 98.3 percent.", source_id)
        add_pair(pairs, "How many active for-sale listings were there in Iowa City?",
                 "There were 248 active for-sale listings in Iowa City.", source_id)
        add_pair(pairs, "How many new listings were recorded in Iowa City as of February 28, 2026?",
                 "There were 54 new listings in Iowa City as of February 28, 2026.", source_id)
        add_pair(pairs, "What percent of home sales in Iowa City were over list price?",
                 "10.2 percent of home sales in Iowa City were over list price.", source_id)
        add_pair(pairs, "What percent of home sales in Iowa City were under list price?",
                 "68.1 percent of home sales in Iowa City were under list price.", source_id)
        add_pair(pairs, "What was the median number of days to pending in Iowa City?",
                 "The median days to pending in Iowa City was 56 days.", source_id)
        add_pair(pairs, "What does a 0.983 sale-to-list ratio imply for buyers in Iowa City?",
                 "It implies buyers were negotiating about 1.7 percent below asking price on average.", source_id)
        add_pair(pairs, "Which was higher in early 2026 in Iowa City, the median list price or the median sale price?",
                 "The median list price was higher. It was $297,567 compared with a median sale price of $289,417.", source_id)
        add_pair(pairs, "Were most home sales in Iowa City closing above or below list price?",
                 "Most home sales were closing below list price, since 68.1 percent of sales were under list and 10.2 percent were over list.", source_id)

    elif source_id == "housing_zillow_002":
        add_pair(pairs, "Which Iowa City neighborhood had the highest Zillow Home Value Index in February 2026?",
                 "South Pointe had the highest Zillow Home Value Index at $314,570.", source_id)
        add_pair(pairs, "Which Iowa City neighborhood had the lowest tracked Zillow Home Value Index in February 2026?",
                 "Broadway had the lowest tracked Zillow Home Value Index at $96,298.", source_id)
        add_pair(pairs, "What was the Zillow Home Value Index for Pepperwood in February 2026?",
                 "Pepperwood had a Zillow Home Value Index of $283,676 in February 2026.", source_id)
        add_pair(pairs, "What was the Zillow Home Value Index for Oak Grove in February 2026?",
                 "Oak Grove had a Zillow Home Value Index of $242,293 in February 2026.", source_id)
        add_pair(pairs, "What was the Zillow Home Value Index for Wetherby in February 2026?",
                 "Wetherby had a Zillow Home Value Index of $241,441 in February 2026.", source_id)
        add_pair(pairs, "What was the Zillow Home Value Index for Creekside in February 2026?",
                 "Creekside had a Zillow Home Value Index of $232,169 in February 2026.", source_id)
        add_pair(pairs, "What was the Zillow Home Value Index for Lucas Farms in February 2026?",
                 "Lucas Farms had a Zillow Home Value Index of $223,431 in February 2026.", source_id)
        add_pair(pairs, "What was the Zillow Home Value Index for Grant Wood in February 2026?",
                 "Grant Wood had a Zillow Home Value Index of $218,543 in February 2026.", source_id)
        add_pair(pairs, "Did the neighborhood breakdown report enough data for Paddock?",
                 "No. Paddock was listed as not available because there was insufficient data.", source_id)
        add_pair(pairs, "Approximately how much more expensive was South Pointe than Broadway according to the neighborhood breakdown?",
                 "South Pointe commanded roughly a 44 percent premium over Broadway.", source_id)
        add_pair(pairs, "What explanation does the source give for Broadway's low home value index?",
                 "The source suggests Broadway's low value likely reflects a high share of multi-family and student housing.", source_id)
        add_pair(pairs, "Which neighborhoods were described as clustering in the $218,000 to $242,000 range?",
                 "Grant Wood, Wetherby, and Creekside were described as clustering in the $218,000 to $242,000 range.", source_id)

    elif source_id == "housing_zillow_003":
        add_pair(pairs, "What was the average rent in Iowa City as of February 28, 2026?",
                 "The average rent in Iowa City was $1,308 per month as of February 28, 2026.", source_id)
        add_pair(pairs, "What was the national average rent in the Zillow rent comparison?",
                 "The national average rent in the comparison was $1,895 per month.", source_id)
        add_pair(pairs, "What was the month-over-month rent change in Iowa City?",
                 "The month-over-month rent change in Iowa City was plus 0.4 percent.", source_id)
        add_pair(pairs, "What was the year-over-year rent change in Iowa City?",
                 "The year-over-year rent change in Iowa City was plus 4.3 percent.", source_id)
        add_pair(pairs, "How did Iowa City average rent compare with the national average?",
                 "Iowa City average rent was approximately 31 percent below the national average.", source_id)
        add_pair(pairs, "Why does the source say Iowa City rents are below the national average?",
                 "The source attributes lower rents to Iowa's generally lower cost of living and the dominance of student rental housing, which increases supply near the University of Iowa campus.", source_id)
        add_pair(pairs, "What does the source suggest about the 4.3 percent year-over-year rent increase?",
                 "The source suggests the increase reflects continued demand pressure in the rental market despite the availability of affordable for-sale options.", source_id)
        add_pair(pairs, "Why does the source say the Zillow Observed Rent Index is a cleaner measure than raw asking-price averages?",
                 "The source says ZORI controls for changes in the quality of available rental inventory, making it a cleaner measure of rent-level trends than raw asking-price averages.", source_id)
        add_pair(pairs, "How much lower was Iowa City average rent than the national average in dollar terms?",
                 "Iowa City average rent was $587 lower per month than the national average.", source_id)

    return pairs

def flat_to_messages(example):
    """Convert flat example to ChatML message format."""
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": example["output"]},
        ],
        "source_id": example["source_id"]
    }

# Load and prepare data
records = []
with open(HOUSING_SOURCE_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

flat_examples = []
for record in records:
    flat_examples.extend(generate_pairs_from_record(record))

message_examples = [flat_to_messages(ex) for ex in flat_examples]

# Save datasets
Path(FLAT_OUTPUT_FILE).parent.mkdir(parents=True, exist_ok=True)

with open(FLAT_OUTPUT_FILE, "w", encoding="utf-8") as f:
    for ex in flat_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

with open(MESSAGES_OUTPUT_FILE, "w", encoding="utf-8") as f:
    for ex in message_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print(f"✓ Prepared {len(flat_examples)} Q&A pairs")
print(f"  - Flat format: {FLAT_OUTPUT_FILE}")
print(f"  - Message format: {MESSAGES_OUTPUT_FILE}")

Generating train split: 3 examples [00:00, 355.03 examples/s]

Wrote 34 flat QA examples to finetune_qa.jsonl
Wrote 34 message-format examples to finetune_messages.jsonl


## Step 2: Fine-tune with QLoRA

Fine-tune the base model using 4-bit quantization + LoRA for memory efficiency.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
import torch

print(f"Loading base model: {BASE_MODEL_ID}")

# Load dataset in message format
hgdataset = load_dataset("json", data_files=MESSAGES_OUTPUT_FILE, split="train")
print(f"✓ Loaded {len(hgdataset)} training examples")

# ── 4-BIT QUANTIZATION WITH QLORA ──────────────────────────────
# Reduces memory footprint significantly while maintaining performance
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",  # Normal Float 4
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

# ── LORA CONFIGURATION ─────────────────────────────────────────
# Target attention layers for parameter-efficient fine-tuning
lora_config = LoraConfig(
    r=16,                     # LoRA rank
    lora_alpha=32,            # LoRA scaling factor
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
print(f"✓ Model prepared with LoRA")
model.print_trainable_parameters()

# ── TRAINING CONFIGURATION ────────────────────────────────────
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    bf16=True,  # Use bfloat16 for faster training on compatible GPUs
    logging_steps=10,
    save_steps=100,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_seq_length=MAX_SEQ_LENGTH,
)

print(f"Starting training...")
trainer = SFTTrainer(
    model=model,
    train_dataset=hgdataset,
    args=training_args,
)

trainer.train()
print(f"✓ Training complete. Model saved to: {OUTPUT_DIR}")

Generating train split: 34 examples [00:00, 34454.30 examples/s]


trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750


Truncating train dataset: 100%|██████████| 34/34 [00:00<00:00, 4412.73 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.
/Users/siqili/micromamba/envs/my_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


TrainOutput(global_step=9, training_loss=3.3152103424072266, metrics={'train_runtime': 113.5898, 'train_samples_per_second': 0.898, 'train_steps_per_second': 0.079, 'total_flos': 57948713410560.0, 'train_loss': 3.3152103424072266})

## Step 3: Merge & Save

Merge the LoRA weights into the base model for an inference-ready model.

In [ ]:
from peft import PeftModel

print(f"Merging LoRA weights into base model...")

# Merge LoRA weights into the base model
# This combines the learned parameters with the original model weights
merged_model = model.merge_and_unload()

# Save the merged model
print(f"Saving merged model to: {MERGED_MODEL_DIR}")
merged_model.save_pretrained(MERGED_MODEL_DIR)
tokenizer.save_pretrained(MERGED_MODEL_DIR)

print(f"✓ Merged model saved locally")

# Optional: Push to Hugging Face Hub
# Uncomment the lines below to upload to Hub (requires authentication)
# print(f"Pushing to Hub: {HF_MODEL_REPO}")
# merged_model.push_to_hub(HF_MODEL_REPO)
# tokenizer.push_to_hub(HF_MODEL_REPO)
# print(f"✓ Model pushed to Hub")

/Users/siqili/micromamba/envs/my_env/lib/python3.11/site-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
Processing Files (1 / 1): 100%|██████████| 1.07GB / 1.07GB,  376MB/s  
New Data Upload: 100%|██████████| 12.7MB / 12.7MB, 5.30MB/s  
Processing Files (1 / 1): 100%|██████████| 17.2MB / 17.2MB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/SiwgiLi/llama3-housing-lora-finetuned/commit/0cb97b6e4023916a7a193ad953de6a621287ecf6', commit_message='Upload tokenizer', commit_description='', oid='0cb97b6e4023916a7a193ad953de6a621287ecf6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/SiwgiLi/llama3-housing-lora-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='SiwgiLi/llama3-housing-lora-finetuned'), pr_revision=None, pr_num=None)